# M2.2.a: ClusterNo — per-building shape clustering

Implements ClusterNo with numeric alignment target vs buds-lab Feature generator Cells 8-10.
Input: train_features.csv + test_features.csv  
Output: data/interim/clusterno.csv (406 buildings × ClusterNo label)

Preprocessing chain (order matters): (z-score + ±10σ clip) × 2 → log1p → transpose → fillna(0) → StandardScaler → KMeans

In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

train_features = pd.read_csv("../data/raw/train_features.csv")
test_features = pd.read_csv("../data/raw/test_features.csv")
print(f"Train: {train_features.shape}")
print(f"Test:  {test_features.shape}")

Train: (1749494, 57)
Test:  (1800567, 57)


In [2]:
merged = pd.concat([train_features, test_features], axis=0, ignore_index=True)
pivot = merged.pivot_table(
    index="timestamp", columns="building_id", values="meter_reading"
)
print(f"Pivot shape: {pivot.shape}")  # expected (8784, 406)
print(f"NaN ratio: {pivot.isna().sum().sum() / pivot.size:.2%}")

Pivot shape: (8784, 406)
NaN ratio: 6.13%


In [3]:
# Order matters: z-score+clip FIRST, log1p comes later
for i in range(2):
    pivot = (pivot - pivot.mean()) / pivot.std()
    pivot = pivot[pivot < 10]
    pivot = pivot[pivot > -10]
    nan_ratio = pivot.isna().sum().sum() / pivot.size
    vmin = pivot.min().min()
    vmax = pivot.max().max()
    print(
        f"After z-score+clip round {i + 1}: shape={pivot.shape}, "
        f"NaN ratio={nan_ratio:.2%}, "
        f"value range=[{vmin:.2f}, {vmax:.2f}]"
    )

After z-score+clip round 1: shape=(8784, 406), NaN ratio=6.14%, value range=[-9.95, 9.85]


After z-score+clip round 2: shape=(8784, 406), NaN ratio=6.14%, value range=[-9.95, 9.87]


In [4]:
pivot = np.log1p(pivot)
nan_ratio = pivot.isna().sum().sum() / pivot.size
print(f"After log1p: NaN ratio={nan_ratio:.2%}")
print(f"Value range: [{pivot.min().min():.2f}, {pivot.max().max():.2f}]")

After log1p: NaN ratio=15.48%
Value range: [-11.65, 2.39]


C:\Users\tonykuo\projects\lead-reproduction\.venv\Lib\site-packages\pandas\core\internals\blocks.py:347: RuntimeWarning: invalid value encountered in log1p
  result = func(self.values, **kwargs)


In [5]:
df_buildings = pivot.T  # shape (406, 8784)
print(f"Transposed shape: {df_buildings.shape}")

scaler = StandardScaler()
X_cluster = scaler.fit_transform(df_buildings.fillna(0))
print(f"X_cluster shape: {X_cluster.shape}")
print(f"X_cluster has NaN: {np.isnan(X_cluster).any()}")

Transposed shape: (406, 8784)


X_cluster shape: (406, 8784)
X_cluster has NaN: False


In [6]:
km = KMeans(n_clusters=10, max_iter=10000, random_state=666, n_init=10)
labels = km.fit_predict(X_cluster)
print(f"Labels shape: {labels.shape}")  # (406,)
print("Label distribution:")
print(pd.Series(labels).value_counts().sort_index())
print(f"\nKMeans inertia: {km.inertia_:.4f}")
print(f"KMeans n_iter_: {km.n_iter_}")  # should be < 10000

Labels shape: (406,)
Label distribution:
0    36
1    90
2     9
3    22
4    28
5    33
6     3
7    63
8    80
9    42
Name: count, dtype: int64

KMeans inertia: 2598263.4537
KMeans n_iter_: 16


In [7]:
building_ids = df_buildings.index.tolist()  # same order as pivot.columns
cluster_df = pd.DataFrame({"building_id": building_ids, "ClusterNo": labels})
print(f"Cluster_df shape: {cluster_df.shape}")  # (406, 2)
print(cluster_df.head())
print(cluster_df.tail())

Cluster_df shape: (406, 2)
   building_id  ClusterNo
0            1          8
1           18          8
2           19          8
3           26          8
4           32          8
     building_id  ClusterNo
401         1322          4
402         1323          7
403         1353          8
404         1384          8
405         1425          8


In [8]:
train_buildings = train_features["building_id"].unique()
train_clusters = cluster_df[cluster_df["building_id"].isin(train_buildings)]
print(f"Train building count: {len(train_clusters)}")  # expected 200
print("\nTrain cluster distribution:")
print(train_clusters["ClusterNo"].value_counts().sort_index())

Train building count: 200

Train cluster distribution:
ClusterNo
0    19
1    44
2     2
3    10
4    18
5    14
6     2
7    32
8    37
9    22
Name: count, dtype: int64


In [9]:
import os

os.makedirs("../data/interim", exist_ok=True)
cluster_df.to_csv("../data/interim/clusterno.csv", index=False)
print("Saved to data/interim/clusterno.csv")
print(cluster_df.dtypes)

Saved to data/interim/clusterno.csv
building_id    int64
ClusterNo      int32
dtype: object
